# Experiment 9 — Retrieval-Augmented Generation (RAG)



In [1]:
%pip install -q langchain-text-splitters langchain-chroma langchain-huggingface sentence-transformers transformers torch

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


## Module 2 — Import libraries

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import pipeline

## Module 3 — Create policy document

In [3]:
document_text = """
Acme Corp Remote Work Policy (2025/2026):
Employees are eligible for full remote work after completing 90 days of continuous employment.
Remote staff are provided an initial home office setup stipend of $750.
Core working hours are strictly required between 10:00 AM and 3:00 PM Eastern Standard Time (EST).
Expense reimbursements for internet and utility costs are capped at $100 per month.
All security compliance protocols must be passed bi-annually.
"""
print(document_text)


Acme Corp Remote Work Policy (2025/2026):
Employees are eligible for full remote work after completing 90 days of continuous employment.
Remote staff are provided an initial home office setup stipend of $750.
Core working hours are strictly required between 10:00 AM and 3:00 PM Eastern Standard Time (EST).
Expense reimbursements for internet and utility costs are capped at $100 per month.
All security compliance protocols must be passed bi-annually.



## Module 4 — Split text into chunks

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
chunks = text_splitter.split_text(document_text)
print("Number of chunks:", len(chunks))
for i, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {i}:\n{chunk}")

Number of chunks: 4

Chunk 1:
Acme Corp Remote Work Policy (2025/2026):
Employees are eligible for full remote work after completing 90 days of continuous employment.

Chunk 2:
Remote staff are provided an initial home office setup stipend of $750.

Chunk 3:
Core working hours are strictly required between 10:00 AM and 3:00 PM Eastern Standard Time (EST).

Chunk 4:
Expense reimbursements for internet and utility costs are capped at $100 per month.
All security compliance protocols must be passed bi-annually.


## Module 5 — Create sentence embeddings

In [5]:
embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded successfully.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Embedding model loaded successfully.


## Module 6 — Store chunks in Chroma vector database

In [6]:
vector_db = Chroma.from_texts(texts=chunks, embedding=embedding_function)
print("Chroma vector database created successfully.")

Chroma vector database created successfully.


In [8]:
%pip install transformers==4.57.6 sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 72.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.0 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency con

## Module 7 — Load the answer-generation model

In [7]:
qa_generator = pipeline("text2text-generation", model="google/flan-t5-base")
print("Text generation pipeline loaded successfully.")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Text generation pipeline loaded successfully.


## Module 8 — Define the RAG question-answering function

In [8]:
def answer_question_with_rag(query):
    # Retrieve relevant context
    retrieved_docs = vector_db.similarity_search(query, k=2)
    context = "\n".join([doc.page_content for doc in retrieved_docs])

    # Construct context-augmented prompt
    augmented_prompt = (
        f"Answer the user question using ONLY the provided context.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )

    # Generate answer
    result = qa_generator(augmented_prompt, max_length=100)
    return result[0]["generated_text"], context

print("RAG question-answering function defined successfully.")

RAG question-answering function defined successfully.


## Module 9 — Ask a question

In [9]:
user_query = "What is the home office setup stipend limit?"
answer, context_used = answer_question_with_rag(user_query)
print(f"QUERY: {user_query}")
print(f"\nRETRIEVED CONTEXT:\n{context_used}")
print(f"\nGENERATED ANSWER:\n{answer}")

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: What is the home office setup stipend limit?

RETRIEVED CONTEXT:
Remote staff are provided an initial home office setup stipend of $750.
Expense reimbursements for internet and utility costs are capped at $100 per month.
All security compliance protocols must be passed bi-annually.

GENERATED ANSWER:
$750
